# Exploration: signals, peaks and weights

Loads the **cached** results of one pipeline run from `outputs/<VIDEO_STEM>/` (no Whisper, no video decoding) and shows how the three signals combine into the final score.

Run the pipeline first (`cli.py` or the app), then run this notebook top to bottom with the `.venv` kernel.

In [1]:
VIDEO_STEM = "test_match"                     # folder name under outputs/
TRUTH_CSV = "samples/truth_test.csv"          # optional: true event times (time,event); skipped if missing

In [2]:
import os
import sys
from pathlib import Path

# work from the project root whether the notebook is started there or in notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter, MultipleLocator

from evaluate import compute_metrics, load_truth
from src.scorer import combine_signals, find_highlights
from src.ui_style import EVENT_COLORS
from src.utils import fmt_time, load_config

OUT = ROOT / "outputs" / VIDEO_STEM
cfg = load_config()
print("Output folder:", OUT)

Output folder: C:\projects\fifa-highlight-detector\outputs\test_match


## Load cached tables

In [3]:
signals = pd.read_csv(OUT / "signals.csv", keep_default_na=False)
highlights = pd.read_csv(OUT / "highlights.csv", keep_default_na=False)
audio_df = pd.read_csv(OUT / "cache" / "audio_features.csv")
kw_df = pd.read_csv(OUT / "cache" / "keywords.csv", keep_default_na=False)  # keyword_hits '' not NaN
video_df = pd.read_csv(OUT / "cache" / "video_features.csv")
truth = load_truth(TRUTH_CSV) if Path(TRUTH_CSV).is_file() else []

print(f"{len(signals)} seconds ({fmt_time(len(signals))}), {len(highlights)} highlights, "
      f"{len(truth)} true events")
highlights[["rank", "timestamp", "score", "event_type", "reason"]]

1800 seconds (30:00), 10 highlights, 2 true events


,rank,timestamp,score,event_type,reason
0,1,11:38,0.6453,Card / Foul,crowd spike (audio z=6.0) + 'penalty' + 'goal'...
1,2,04:47,0.4448,Card / Foul,crowd spike (audio z=2.5) + 'penalty' + replay...
2,3,12:37,0.4235,Card / Foul,crowd spike (audio z=2.8) + 'penalty' x2 + 'go...
3,4,06:01,0.4101,Goal,crowd spike (audio z=2.5) + confirmed by 'it's...
4,5,28:36,0.4086,Card / Foul,crowd spike (audio z=3.7) + 'penalty' + replay...
5,6,25:35,0.3992,Goal,crowd spike (audio z=1.9) + confirmed by 'scor...
6,7,21:45,0.3989,Exciting Moment,crowd spike (audio z=6.0)
7,8,27:52,0.3911,Big Chance / Save,crowd spike (audio z=2.9) + 'off the underside...
8,9,13:31,0.3887,Exciting Moment,crowd spike (audio z=2.5) + replay detected
9,10,19:18,0.3762,Card / Foul,crowd spike (audio z=2.3) + 'yellow card' + re...


## Signals on one time axis

Grey dashed lines = true events from the truth file, dots = the run's highlight peaks (colored by event type, same colors as the app).

In [4]:
mmss = FuncFormatter(lambda x, _: fmt_time(x))


def mark_truth(ax):
    for t in truth:
        ax.axvline(t["time_s"], color="0.35", ls="--", lw=1)


panels = [("audio_z", "#2563EB"), ("keyword_score", "#F59E0B"), ("video_z", "#8B5CF6"),
          ("score", "#16A34A")]
fig, axes = plt.subplots(len(panels), 1, figsize=(13, 8.5), sharex=True, dpi=80)
for ax, (col, color) in zip(axes, panels):
    ax.plot(signals["second"], signals[col], color=color, lw=0.8)
    ax.set_ylabel(col)
    ax.grid(alpha=0.3)
    mark_truth(ax)
for _, h in highlights.iterrows():
    axes[-1].scatter(h["peak_s"], h["score"], s=60, zorder=3, edgecolor="white",
                     color=EVENT_COLORS.get(h["event_type"], "#8B5CF6"))
    axes[-1].annotate(f"#{h['rank']}", (h["peak_s"], h["score"]), textcoords="offset points",
                      xytext=(0, 6), ha="center", fontsize=8)
axes[-1].xaxis.set_major_formatter(mmss)
axes[-1].xaxis.set_major_locator(MultipleLocator(300))  # a tick every 5 min
axes[-1].set_xlabel("match time (mm:ss)")
fig.suptitle(f"{VIDEO_STEM}: signals, highlight peaks and true events")
fig.tight_layout()
plt.show()

## Try other weights

Change `WEIGHTS` and re-run this cell: it calls `combine_signals` + `find_highlights` from `src.scorer` on the cached tables (nothing is written, config.yaml is not changed). Weights are normalized to sum 1.

In [5]:
WEIGHTS = {"audio": 0.5, "keywords": 0.35, "video": 0.15}   # config default; try e.g. audio 0.3

sc = {**cfg["scoring"], "weights": WEIGHTS}
new = combine_signals(audio_df, kw_df, video_df, WEIGHTS, sc["smooth_window_s"], sc)
peaks = find_highlights(new["score"], int(sc["top_n"]), int(sc["min_peak_distance_s"]),
                        float(sc.get("min_prominence", 0.02)))
base_peaks = [int(p) for p in highlights.sort_values("rank")["peak_s"]]

fig, ax = plt.subplots(figsize=(13, 3.2), dpi=80)
ax.plot(signals["second"], signals["score"], color="0.6", lw=0.8, label="saved run")
ax.plot(new["second"], new["score"], color="#16A34A", lw=0.9, label=f"weights {WEIGHTS}")
ax.scatter(peaks, new["score"].iloc[peaks], color="#16A34A", zorder=3, label="new peaks")
mark_truth(ax)
ax.xaxis.set_major_formatter(mmss)
ax.xaxis.set_major_locator(MultipleLocator(300))
ax.grid(alpha=0.3)
ax.legend(loc="upper left", fontsize=8)
plt.show()

print("saved run peaks:", ", ".join(fmt_time(p) for p in base_peaks))
print("new peaks:      ", ", ".join(fmt_time(p) for p in peaks))
if truth:
    # labels are not recomputed here, so only time matching (precision/recall) is meaningful
    pred = [{"rank": i, "peak_s": float(p), "event_type": ""} for i, p in enumerate(peaks, 1)]
    r = compute_metrics(truth, pred, tolerance=20)
    print(f"vs truth (+/-20 s): {r['tp']}/{r['n_truth']} found, ranks "
          f"{[p['rank'] for _, p, _ in r['matches']]}")

saved run peaks: 11:38, 04:47, 12:37, 06:01, 28:36, 25:35, 21:45, 27:52, 13:31, 19:18
new peaks:       11:38, 04:47, 12:37, 06:01, 28:36, 25:35, 21:45, 27:52, 13:31, 19:18
vs truth (+/-20 s): 2/2 found, ranks [4, 6]
